# BetaTour: POPMUSIC'i gerçekten tamamlayan 5+1 deneyi

Önceki 4+1 deneyinde POPMUSIC'in beşinci adayı çıkarılıyordu. Bu deney her düğümün **beş POPMUSIC adayının tamamını** korur; altıncı aday olarak POPMUSIC-5 grafiğinde iki yönde de bulunmayan bir kenar ekler. Kenarı BetaTour, tur sıklığı veya geometrik yakınlık seçer. Kesintisiz POPMUSIC-5 ve POPMUSIC-6 aynı toplam süre bütçesiyle pratik karşılaştırmadır.

**Çalıştırma:** Google Colab'da A100 ya da L4 GPU, High RAM ve en az 12 görünür vCPU seçin; ardından `Çalışma zamanı > Tümünü çalıştır` deyin. Drive erişimine izin verin. Sonuçlar `MyDrive/BetaTour/augment40_v1/` klasörüne yazılır. Tamamlanmış örnekler yeniden çalışmaz; yeni bir sanal makinede yarım kalan yalnız o örnek baştan çalışır. Yaklaşık **14–17 saat** planlayın; kesintiler bu süreyi uzatabilir.

Tamamlanınca küçük `augment40_v1_results.zip` tarayıcıya otomatik iner. Aday ve ilk-tur dizilerini içeren büyük `augment40_v1_audit.zip` Drive'da kalır. Her iki ZIP kapanış ve CRC denetiminden sonra atomik yayımlanır.


## 1. Önceden sabitlenen deney ve analiz planı

Önceki residual40 örnekleri bu doğrulamada kullanılmaz. 24 adet 10.000 düğümlü ve 16 adet 20.000 düğümlü yeni sentetik örnek oluşturulur. Her örnek için altı LKH tohumu kullanılır; sonuç, aynı süre sınırı içindeki en iyi altı turun en düşük maliyetidir.

**İki birincil pratik kıyas:** BetaTour 5+1 / kesintisiz POPMUSIC-5 ve BetaTour 5+1 / kesintisiz POPMUSIC-6. Pratik başarı kararı ancak **ikisinde birden** iki yönlü kesin işaret testi p<0,05, daha çok BetaTour galibiyeti ve negatif ortalama eşli yüzde fark varsa verilir. İkincil kıyaslar yeniden başlatılmış POPMUSIC-5, en yakın yeni kenar 5+1 ve tur sıklığı 5+1'dir; üç ikincil p-değeri Holm ile düzeltilir. Alt gruplar betimleyicidir. Sonuçlara bakarak aday sayısı veya α sırası değiştirilmez; böyle değişiklikler yeni örnek gerektirir.


In [ ]:
# Colab: A100 or L4 GPU, High RAM and at least 12 visible vCPUs.
# Keep these settings fixed after the first instance has been saved.
EXPERIMENT_NAME = "augment40_v1"
PROFILE = "augment40"
GROUPS = ["uni10k", "clu10k", "uni20k", "clu20k"]
INSTANCE_SPECS = [
    ["uni10k", 10000, 12, "uniform", 4107, 120.0],
    ["clu10k", 10000, 12, "clustered", 4208, 120.0],
    ["uni20k", 20000, 8, "uniform", 4309, 300.0],
    ["clu20k", 20000, 8, "clustered", 4410, 300.0],
]
MAX_NEW_INSTANCES = None  # e.g. 3 stops after three new instances this session
SAVE_TO_DRIVE = True
INCLUDE_ARRAYS_IN_ZIP = True  # phase1 and candidate NPZ files permit independent checks

W = 6
REQUIRE_12_VCPUS = True
SEEDS = list(range(301, 307))
K_GRAPH = 16
K_POP5 = 5
K_AUG6 = 6
PAD_SECONDS = 3.0
REPO_COMMIT = "d63f7e17942a6af5cba14ad010edca1069d477b3"
CHECKPOINT_SHA256 = "2322f66c752913f6179f8c61f6b50b0f9d21426857d3ec0155e02891fb3ebd70"
ARM_NAMES = ["pop5_full", "pop6_full", "restart_pop_5",
             "augment_nearest5p1", "augment_frequency5p1", "augment_betatour5p1"]

assert W == len(SEEDS) and K_POP5 == 5 and K_AUG6 == 6 and K_GRAPH == 16
assert PROFILE == "augment40" and sum(spec[2] for spec in INSTANCE_SPECS) == 40

ANALYSIS_PLAN = {
    "unit": "instance",
    "sample_size": 40,
    "endpoint": "lowest tour cost among six seeds by each instance's wall-clock deadline",
    "primary": ["augment_betatour5p1 vs pop5_full", "augment_betatour5p1 vs pop6_full"],
    "primary_test": "both two-sided exact sign tests on non-tied instance pairs must have p<0.05 and BetaTour lower mean cost; intersection-union rule",
    "effect": "mean paired 100*(augment_betatour5p1-control)/control",
    "secondary": ["augment_betatour5p1 vs restart_pop_5",
                  "augment_betatour5p1 vs augment_nearest5p1",
                  "augment_betatour5p1 vs augment_frequency5p1"],
    "multiplicity": "co-primary success requires both practical comparisons; Holm over three secondary overall comparisons",
    "subgroups": "10k/20k and uniform/clustered descriptive only",
    "post_result_tuning": "none; candidate-count or alpha-order changes require a new holdout",
}


## 2. Ortam, model, Drive ve LKH

Kaynak commit'i ve model SHA256 değeri sabittir. LKH-3.0.14 resmi kaynaktan, o erişilemezse SHA256 denetimli yedekten derlenir; indirilen kaynak ve ikili SHA256 değeri Drive'a kaydedilir. `BetaTour/augment40_v1` klasörü otomatik açılır. Manifest ayarların yeniden çalıştırmada değişmediğini, GPU türü ve vCPU sayısının aynı kaldığını denetler.


In [ ]:
import hashlib
import importlib.util
import json
import math
import os
import platform
import subprocess
import sys
import tarfile
import time
import urllib.request
import uuid
from pathlib import Path

for package, module in [("scipy", "scipy"), ("pandas", "pandas"), ("tqdm", "tqdm")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import scipy
import torch

torch.set_num_threads(1)
assert torch.cuda.is_available(), "Colab'da GPU runtime seçin: L4 veya A100."
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
cpu_max = Path("/sys/fs/cgroup/cpu.max")
if cpu_max.exists():
    quota, period = cpu_max.read_text().split()[:2]
    if quota != "max":
        CPU_COUNT = min(CPU_COUNT, max(1, math.floor(int(quota) / int(period))))
assert CPU_COUNT >= W, f"Altı LKH süreci için en az {W} vCPU gerekli; görünen: {CPU_COUNT}"
if REQUIRE_12_VCPUS:
    assert CPU_COUNT >= 2 * W, f"Makaledeki 12 vCPU düzeni için 12 vCPU gerekli; görünen: {CPU_COUNT}"
if CPU_COUNT < 2 * W:
    print(f"UYARI: {CPU_COUNT} vCPU var. Bu çalışma makaledeki 12 vCPU düzenini tekrarlamıyor.")

def cpu_model_name():
    path = Path("/proc/cpuinfo")
    if path.exists():
        for line in path.read_text().splitlines():
            if line.lower().startswith("model name"):
                return line.split(":", 1)[1].strip()
    return platform.processor() or "unknown"

CPU_MODEL = cpu_model_name()

WORK_ROOT = Path("/content/betatour_validation_work")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO = WORK_ROOT / "repo"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", "https://github.com/kurbankotan/betatour-tsp-lkh-candidates.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", REPO_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_COMMIT], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == REPO_COMMIT
sys.path.insert(0, str(REPO / "src"))
import tsplab as T
import tsp_model as M

checkpoint_path = REPO / "checkpoints" / "betatour_v2.pt"
actual_hash = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
assert actual_hash == CHECKPOINT_SHA256, "Model dosyası beklenen SHA256 değeriyle eşleşmiyor."
MODEL, CK = M.load_model(str(checkpoint_path), DEVICE)  # pinned, trusted repository
assert MODEL.cfg.get("use_ensemble") is True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BETATOUR = Path("/content/drive/MyDrive/BetaTour")
    DRIVE_BETATOUR.mkdir(parents=True, exist_ok=True)
    OUTPUT_ROOT = DRIVE_BETATOUR / EXPERIMENT_NAME
else:
    OUTPUT_ROOT = WORK_ROOT / "results" / EXPERIMENT_NAME
for sub in ["instances", "stages", "candidates", "errors", "summary", "references"]:
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)

HARDWARE = {
    "gpu": GPU_NAME,
    "vcpus_visible": CPU_COUNT,
    "cpu_model": CPU_MODEL,
    "ram_gb": T.env_info().get("ram_gb"),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
}
MANIFEST = {
    "schema": 3,
    "experiment_name": EXPERIMENT_NAME,
    "profile": PROFILE,
    "groups": GROUPS,
    "instance_specs": INSTANCE_SPECS,
    "augmentation_rule": {
        "pop_slots": K_POP5,
        "total_slots": K_AUG6,
        "novelty": "undirected edge absent from seed's entire filled POP5 graph",
        "model_choice": "first novel edge in model rank",
        "frequency_choice": "first novel edge in tour-frequency rank",
        "nearest_choice": "closest novel geometric neighbour",
        "position": "sixth; alpha defaults to 500 in candidate file",
        "fallback": "nearest novel edge among 64/128 neighbours, then all nodes if necessary",
    },
    "seeds": SEEDS,
    "workers": W,
    "graph_k": K_GRAPH,
    "pop_candidate_k": K_POP5,
    "augmented_candidate_k": K_AUG6,
    "repo_commit": REPO_COMMIT,
    "checkpoint_sha256": CHECKPOINT_SHA256,
    "hardware": HARDWARE,
    "arms": ARM_NAMES,
    "analysis_plan": ANALYSIS_PLAN,
}
manifest_path = OUTPUT_ROOT / "manifest.json"
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text())
    for key in ["schema", "profile", "groups", "instance_specs", "augmentation_rule", "seeds", "workers", "graph_k",
                "pop_candidate_k", "augmented_candidate_k", "repo_commit", "checkpoint_sha256", "arms", "analysis_plan"]:
        assert saved[key] == MANIFEST[key], f"{key} değişti. Ayrı EXPERIMENT_NAME kullanın."
    for key in ["gpu", "vcpus_visible"]:
        assert saved["hardware"][key] == HARDWARE[key], \
            f"Donanım {key} değişti. Eşli süre karşılaştırması için aynı Colab türünü seçin."
    if saved["hardware"] != HARDWARE:
        print("UYARI: CPU modeli, RAM veya kütüphane sürümleri değişmiş; her örneğin donanımı ayrıca kaydedilecek.")
else:
    manifest_tmp = manifest_path.with_name("manifest.json.tmp")
    manifest_tmp.write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))
    os.replace(manifest_tmp, manifest_path)

# This local marker survives a notebook rerun on the same Colab VM. A fresh VM
# gets a new marker: incomplete instances are then rerun on one machine.
session_path = WORK_ROOT / "runtime_session_id.txt"
if session_path.exists():
    SESSION_ID = session_path.read_text().strip()
else:
    SESSION_ID = uuid.uuid4().hex
    session_path.write_text(SESSION_ID)

def get_lkh_3014():
    """Official source first; pinned SHA256 checked source mirror if it is unavailable."""
    root = WORK_ROOT / "lkh"
    binary = root / "LKH-3.0.14" / "LKH"
    if binary.exists():
        return str(binary)
    root.mkdir(parents=True, exist_ok=True)
    options = [
        ("resmi", "https://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.14.tgz", None, 12),
        ("SHA256 denetimli yedek", "https://raw.githubusercontent.com/ZhekaiChen/ALCO/main/third_party/LKH3/LKH-3.0.14.tar",
         "4c4b9e6a77e50a1eb1eb2c8cb384ed5ef64e0dc88fdbf37bd693798469e1cf33", 90),
    ]
    last_error = None
    for label, url, expected_sha, timeout in options:
        archive = root / ("LKH-3.0.14.tar" if url.endswith(".tar") else "LKH-3.0.14.tgz")
        try:
            print("LKH kaynağı:", label, url)
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=timeout) as response, archive.open("wb") as f:
                while True:
                    block = response.read(1024 * 1024)
                    if not block:
                        break
                    f.write(block)
            if expected_sha:
                got = hashlib.sha256(archive.read_bytes()).hexdigest()
                assert got == expected_sha, f"LKH yedek SHA256 uyuşmazlığı: {got}"
            with tarfile.open(archive, "r:*") as tar:
                try:
                    tar.extractall(root, filter="data")
                except TypeError:
                    tar.extractall(root)
            assert (root / "LKH-3.0.14" / "Makefile").exists(), "LKH kaynak klasörü eksik"
            build = subprocess.run(["make", "-j", str(min(4, CPU_COUNT))],
                                   cwd=root / "LKH-3.0.14", stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, timeout=600)
            if build.returncode != 0:
                raise RuntimeError("LKH derlemesi başarısız:\n" + build.stdout[-3000:])
            assert binary.exists(), "LKH ikilisi oluşmadı"
            source_record = root / "build_source.json"
            source_tmp = root / "build_source.json.tmp"
            source_tmp.write_text(json.dumps({
                "source_label": label,
                "source_url": url,
                "source_archive_sha256": hashlib.sha256(archive.read_bytes()).hexdigest(),
                "binary_sha256": hashlib.sha256(binary.read_bytes()).hexdigest(),
            }, indent=2))
            os.replace(source_tmp, source_record)
            return str(binary)
        except Exception as exc:
            last_error = exc
            print(f"{label} başarısız: {exc}")
    raise RuntimeError(f"LKH 3.0.14 indirilemedi/derlenemedi: {last_error}")

LKH = get_lkh_3014()
build_source = WORK_ROOT / "lkh" / "build_source.json"
lkh_audit = json.loads(build_source.read_text()) if build_source.exists() else {}
lkh_audit.update({
    "binary_sha256": hashlib.sha256(Path(LKH).read_bytes()).hexdigest(),
    "session_id": SESSION_ID,
    "lkh_version": "3.0.14",
})
lkh_audit_path = OUTPUT_ROOT / f"lkh_build_{SESSION_ID}.json"
lkh_audit_tmp = lkh_audit_path.with_suffix(".json.tmp")
lkh_audit_tmp.write_text(json.dumps(lkh_audit, indent=2, ensure_ascii=False))
os.replace(lkh_audit_tmp, lkh_audit_path)
print("Hazır:", GPU_NAME, "|", CPU_COUNT, "vCPU | model", sum(p.numel() for p in MODEL.parameters()),
      "parametre | LKH", LKH, "| sonuçlar", OUTPUT_ROOT)


Mounted at /content/drive
LKH kaynağı: resmi https://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.14.tgz
resmi başarısız: <urlopen error timed out>
LKH kaynağı: SHA256 denetimli yedek https://raw.githubusercontent.com/ZhekaiChen/ALCO/main/third_party/LKH3/LKH-3.0.14.tar
Hazır: NVIDIA L4 | 12 vCPU | model 1875787 parametre | LKH /content/betatour_validation_work/lkh/LKH-3.0.14/LKH | sonuçlar /content/drive/MyDrive/BetaTour/augment40_v1


## 3. Yeni örnekler

Üretim tohumları uniform/kümeli 10k için 4107/4208, uniform/kümeli 20k için 4309/4410'dur. LKH tohumları 301–306'dır. Süre sınırı 10k için 120, 20k için 300 saniyedir. Örnekler yalnız sentetik uniform ve kümeli dağılımları kapsar; TSPLIB veya zorlaştırılmış örnekler için ayrıca çalışma gerekir.


In [ ]:
def make_instances():
    specs = INSTANCE_SPECS
    result = []
    for group, n, count, dist, gen_seed, budget in specs:
        rng = np.random.default_rng(gen_seed)
        for i in range(count):
            xy = T.to_int_coords(T.gen_instance(n, dist, rng))
            result.append(dict(group=group, name=f"augment_{group}_{i:03d}", xy=xy,
                               ew="EUC_2D", problem_file=None, budget=budget, optimum=None))
    result.sort(key=lambda x: (x["budget"], x["name"]))
    assert len(result) == 40 and len({x["name"] for x in result}) == 40
    print(f"{len(result)} örnek hazır; tek kol bütçe toplamı {sum(x['budget'] for x in result)/3600:.2f} saat")
    return result

INSTANCES = make_instances()


40 örnek hazır; tek kol bütçe toplamı 2.13 saat


## 4. Altı kollu eşlenik deney

Kollar: kesintisiz POPMUSIC-5, kesintisiz POPMUSIC-6, ortak ilk turlardan yeniden başlayan POPMUSIC-5, POP5 + geometrik en yakın yeni kenar, POP5 + tur sıklığına göre yeni kenar ve POP5 + BetaTour'a göre yeni kenar. Dört yeniden başlayan kol **aynı altı ilk turu** kullanır; bunlar kesintisiz POPMUSIC-5'in ilk turlarıyla denetlenir. POPMUSIC-6'nın ilk turu farklı olabilir.

Üç 5+1 kolunda ilk beş aday sırası aynıdır. Altıncı kenar, ilgili tohumun POPMUSIC-5 aday grafiğinde yönlerden hiçbirinde bulunmaz; aday dosyasında α=500'dür. Hazırlık ve ilk aşama süresi her kolun kendi toplam bütçesinden düşülür. Drive yazımı ve ZIP oluşturma kıyas süresine girmez. Kaydedilen aday dizileri analiz hücresinde doğrulanır.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
import shutil
import traceback

POP_PARAMS_5 = {"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO", "MAX_CANDIDATES": K_POP5}
POP_PARAMS_6 = {"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO", "MAX_CANDIDATES": K_AUG6}

def write_json_atomic(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False))
    os.replace(tmp, path)

def write_npz_atomic(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, path)

def parallel_tasks(func, kwargs):
    with ThreadPoolExecutor(max_workers=W) as pool:
        futures = [pool.submit(func, **kw) for kw in kwargs]
        return [future.result() for future in futures]

def instance_common(inst):
    return dict(lkh_bin=LKH, workroot=str(WORK_ROOT / "lkh_tmp"), xy=inst["xy"],
                problem_file=inst["problem_file"], ew=inst["ew"])

def event_list(res, offset):
    return [[int(cost), float(t)] for cost, t in T.abs_events(res, offset=offset)]

def best_within(events, budget):
    costs = [int(cost) for cost, t in events if t <= budget + 1e-9]
    return min(costs) if costs else None

def make_full_arm(inst, pop_k):
    assert pop_k in (K_POP5, K_AUG6)
    budget = inst["budget"]
    common = instance_common(inst)
    start = time.time()
    params = POP_PARAMS_5 if pop_k == K_POP5 else POP_PARAMS_6
    raw = parallel_tasks(T.task_anytime, [dict(common, budget=budget, seed=seed,
                                             params=params, timeout=budget + PAD_SECONDS)
                                          for seed in SEEDS])
    runs = []
    for seed, res in zip(SEEDS, raw):
        assert res["returncode"] in (0, None) and (res["returncode"] == 0 or res["timed_out"]), (seed, res)
        events = event_list(res, res["t_start"] - start)
        if not events and res["cost_min"] is not None and res["wall"] <= budget:
            events = [[int(res["cost_min"]), float(res["t_start"] - start + res["wall"])]]
        best = best_within(events, budget)
        assert best is not None, f"Kesintisiz POPMUSIC: tohum {seed} için bütçe içinde tur yok"
        runs.append(dict(seed=seed, events=events, best_at_budget=int(best),
                         wall_s=float(res["wall"]), returncode=res["returncode"],
                         timed_out=bool(res["timed_out"])))
    return dict(runs=runs, best_at_budget=min(r["best_at_budget"] for r in runs),
                wall_s=float(time.time() - start), max_candidates=int(pop_k))

def make_phase1(inst):
    common = instance_common(inst)
    start = time.time()
    raw = parallel_tasks(T.task_first_trial, [dict(common, seed=seed, k=K_POP5,
                                                  timeout=inst["budget"] + PAD_SECONDS)
                                               for seed in SEEDS])
    phase_wall = time.time() - start
    n = len(inst["xy"])
    tours, cands, costs, first_events = [], [], [], []
    for seed, res in zip(SEEDS, raw):
        assert T.is_valid_tour(res["tour"], n), f"İlk tur geçersiz: {seed}"
        assert res["cand"].shape == (n, K_POP5), f"Aday boyutu yanlış: {seed}"
        events = event_list(res, res["t_start"] - start)
        if not events:
            events = [[int(res["cost"]), float(res["t_start"] - start + res["wall"])]]
        assert best_within(events, inst["budget"]) is not None, f"İlk tur bütçe dışında: {seed}"
        tours.append(res["tour"])
        cands.append(res["cand"])
        costs.append(res["cost"])
        first_events.append(events)
    arrays = dict(tours=np.stack(tours).astype(np.int32),
                  pop_candidates=np.stack(cands).astype(np.int32),
                  costs=np.asarray(costs, dtype=np.float64))
    meta = dict(wall_s=float(phase_wall), first_events=first_events,
                first_costs=[float(c) for c in costs])
    return arrays, meta

def valid_candidates(cands, n, k=None):
    k = int(cands.shape[2] if k is None else k)
    assert k in (K_POP5, K_AUG6) and cands.shape == (W, n, k), cands.shape
    assert np.all((cands >= 0) & (cands < n)), "Eksik/geçersiz aday bulundu"
    assert not np.any(cands == np.arange(n)[None, :, None]), "Düğüm kendi adayı olamaz"
    for i in range(k):
        for j in range(i + 1, k):
            assert not np.any(cands[:, :, i] == cands[:, :, j]), "Tekrarlı aday bulundu"

def _validate_candidate_rows(candidates, n, width):
    """Validate one candidate array before it is given to LKH."""
    assert candidates.shape == (W, n, width), candidates.shape
    assert np.issubdtype(candidates.dtype, np.integer)
    assert np.all((candidates >= 0) & (candidates < n)), "Missing or invalid candidate"
    assert not np.any(candidates == np.arange(n)[None, :, None]), "Self candidate"
    for a in range(width):
        for b in range(a + 1, width):
            assert not np.any(candidates[:, :, a] == candidates[:, :, b]), "Duplicate candidate"


def _validate_rank(rank, n):
    assert rank.shape == (n, K_GRAPH), rank.shape
    assert np.issubdtype(rank.dtype, np.integer)
    assert np.all((rank >= 0) & (rank < n))
    assert not np.any(rank == np.arange(n)[:, None])
    for a in range(K_GRAPH):
        for b in range(a + 1, K_GRAPH):
            assert not np.any(rank[:, a] == rank[:, b]), "Duplicate graph neighbour"


def _undirected_keys(n, neighbours):
    """Encode an edge identically in both directions."""
    a = np.arange(n, dtype=np.int64)[:, None]
    b = np.asarray(neighbours, dtype=np.int64)
    assert b.ndim == 2 and b.shape[0] == n
    assert np.all((b >= 0) & (b < n)) and not np.any(a == b)
    return np.minimum(a, b) * np.int64(n) + np.maximum(a, b)


def choose_strict_novel(pop5, rank, xy):
    """Keep all five POP entries and add the first undirected POP-novel edge.

    The 16-node rank is method-specific. For a row with no novel graph edge,
    try the nearest 64 and 128 nodes, then scan all nodes. Every fallback is
    timed by the caller and the chosen stage is retained for auditing.
    """
    n = len(xy)
    assert K_POP5 == 5 and K_AUG6 == 6
    _validate_candidate_rows(pop5, n, K_POP5)
    _validate_rank(rank, n)
    rank_keys = _undirected_keys(n, rank)
    out = np.empty((W, n, K_AUG6), dtype=np.int32)
    rank_position = np.zeros((W, n), dtype=np.int16)
    fallback_stage = np.zeros((W, n), dtype=np.uint8)
    rows = np.arange(n)

    for m in range(W):
        old = np.unique(_undirected_keys(n, pop5[m]))
        allowed = ~np.isin(rank_keys, old)
        has = allowed.any(axis=1)
        at = allowed.argmax(axis=1)
        picked = rank[rows, at].copy()
        rank_position[m, has] = (at[has] + 1).astype(np.int16)
        missing = ~has

        if missing.any():
            for width, stage in ((64, 1), (128, 2)):
                near, _ = T.knn(xy, width)
                near_keys = _undirected_keys(n, near)
                available = ~np.isin(near_keys, old)
                found = available.any(axis=1) & missing
                picked[found] = near[rows[found], available.argmax(axis=1)[found]]
                fallback_stage[m, found] = stage
                missing[found] = False
                if not missing.any():
                    break
            if missing.any():
                xy64 = np.asarray(xy, dtype=np.float64)
                for i in np.flatnonzero(missing):
                    dist2 = ((xy64 - xy64[i]) ** 2).sum(axis=1)
                    order = np.argsort(dist2, kind="stable")
                    keys = np.minimum(i, order) * np.int64(n) + np.maximum(i, order)
                    available = (order != i) & ~np.isin(keys, old)
                    assert available.any(), "No POP-novel edge exists for a node"
                    picked[i] = order[np.argmax(available)]
                    fallback_stage[m, i] = 3
                    missing[i] = False

        assert not np.isin(_undirected_keys(n, picked[:, None]), old).any()
        out[m, :, :K_POP5] = pop5[m]
        out[m, :, K_POP5] = picked

    _validate_candidate_rows(out, n, K_AUG6)
    assert np.array_equal(out[:, :, :K_POP5], pop5)
    in_graph = fallback_stage == 0
    meta = {
        "novel_graph_rank_mean": float(rank_position[in_graph].mean()) if in_graph.any() else None,
        "fallback_count": int((fallback_stage != 0).sum()),
        "fallback_fraction": float((fallback_stage != 0).mean()),
        "fallback_64_count": int((fallback_stage == 1).sum()),
        "fallback_128_count": int((fallback_stage == 2).sum()),
        "fallback_fullscan_count": int((fallback_stage == 3).sum()),
    }
    return out, rank_position, fallback_stage, meta


def make_candidate_lists(inst, phase):
    """Return equal POP5 control plus model/frequency/distance 5+1 lists.

    Each arm's recorded overhead is the work it would perform alone after
    the shared first trials. The nearest control uses the geometric 16-NN
    graph already built for POP5 filling.
    """
    xy = inst["xy"]
    n = len(xy)
    tours, raw_pop, costs = phase["tours"], phase["pop_candidates"], phase["costs"]
    assert raw_pop.shape == (W, n, K_POP5)
    assert np.all((raw_pop == -1) | ((raw_pop >= 0) & (raw_pop < n)))
    assert not np.any(raw_pop == np.arange(n)[None, :, None])

    start = time.perf_counter()
    succ = np.stack([T.tour_succ_pred(tour)[0] for tour in tours], axis=1).astype(np.int32)
    successor_s = time.perf_counter() - start

    start = time.perf_counter()
    nearest16, _ = T.knn(xy, K_GRAPH)
    pop5 = np.stack([T.mix_candidates(raw_pop[m], nearest16, k_first=K_POP5,
                                      k=K_POP5, fill=nearest16) for m in range(W)]).astype(np.int32)
    pop5_s = time.perf_counter() - start
    _validate_candidate_rows(pop5, n, K_POP5)

    start = time.perf_counter()
    prediction = M.predict_tourfold(MODEL, xy, K_GRAPH, DEVICE, ens_succ=succ, ens_cost=costs)
    model_predict_s = time.perf_counter() - start
    model_rank = np.asarray(prediction["rank"], dtype=np.int32)
    _validate_rank(model_rank, n)

    start = time.perf_counter()
    xyf = np.asarray(xy, dtype=np.float64)
    span = float((xyf.max(0) - xyf.min(0)).max()) or 1.0
    xyn = (xyf - xyf.min(0)) / span
    graph_nb, graph_d = M.knn_np(xyn, K_GRAPH)
    extra = int(MODEL.cfg.get("graph_extra", 0))
    if extra:
        graph_nb, graph_d, _ = M.extend_graph(xyn, graph_nb, succ, extra)
    graph_s = time.perf_counter() - start
    assert np.array_equal(graph_nb, prediction["NB"]), "Control and model graphs differ"

    start = time.perf_counter()
    frequency = T.ensemble_frequency(graph_nb, succ)
    frequency_rank = T.frequency_candidates(graph_nb, graph_d, frequency, k=K_GRAPH).astype(np.int32)
    frequency_rank_s = time.perf_counter() - start
    _validate_rank(frequency_rank, n)

    nearest_rank = np.asarray(nearest16, dtype=np.int32)
    nearest_rank_s = 0.0
    _validate_rank(nearest_rank, n)

    arrays = {"restart_pop_5": pop5, "model_rank": model_rank,
              "frequency_rank": frequency_rank, "nearest_rank": nearest_rank}
    methods = (
        ("augment_betatour5p1", model_rank, model_predict_s),
        ("augment_frequency5p1", frequency_rank, graph_s + frequency_rank_s),
        ("augment_nearest5p1", nearest_rank, nearest_rank_s),
    )
    meta = {
        "successor_build_s": float(successor_s),
        "pop5_build_s": float(pop5_s),
        "model_predict_wall_s": float(model_predict_s),
        "model_predict_internal_s": float(prediction["t_total"]),
        "control_graph_build_s": float(graph_s),
        "frequency_rank_s": float(frequency_rank_s),
        "nearest_rank_s": float(nearest_rank_s),
        "restart_pop_5_overhead_s": float(pop5_s),
        "raw_pop_candidates_per_node": float((raw_pop >= 0).sum(axis=2).mean()),
        "raw_pop_below_five_fraction": float(((raw_pop >= 0).sum(axis=2) < K_POP5).mean()),
        "pop5_nearest_fill_fraction": float((~(
            pop5[:, :, :, None] == raw_pop[:, :, None, :]).any(-1)).mean()),
    }
    initial_keys = [np.unique(_undirected_keys(n, succ[:, m, None])) for m in range(W)]

    for arm_name, rank, rank_s in methods:
        start = time.perf_counter()
        chosen, graph_pos, fallback_stage, diagnostics = choose_strict_novel(pop5, rank, xy)
        select_s = time.perf_counter() - start
        arrays[arm_name] = chosen
        arrays[arm_name + "_rank_position"] = graph_pos
        arrays[arm_name + "_fallback_stage"] = fallback_stage
        meta[arm_name + "_select_s"] = float(select_s)
        common_s = pop5_s + (successor_s if arm_name != "augment_nearest5p1" else 0.0)
        meta[arm_name + "_overhead_s"] = float(common_s + rank_s + select_s)
        for key, value in diagnostics.items():
            meta[arm_name + "_" + key] = value
        fractions = []
        for m in range(W):
            selected = _undirected_keys(n, chosen[m, :, K_POP5, None]).ravel()
            fractions.append(float(np.isin(selected, initial_keys[m]).mean()))
        meta[arm_name + "_selected_in_own_first_tour_fraction"] = float(np.mean(fractions))

    for arm_name, _, _ in methods:
        assert np.array_equal(arrays[arm_name][:, :, :K_POP5], pop5)
        _validate_candidate_rows(arrays[arm_name], n, K_AUG6)
    return arrays, meta

def make_resumed_arm(inst, phase, phase_meta, candidate_lists, overhead_s, arm_name):
    budget = inst["budget"]
    prefix = float(phase_meta["wall_s"]) + float(overhead_s)
    remaining = budget - prefix
    if remaining <= 0.25:
        runs = []
        for m, seed in enumerate(SEEDS):
            first = phase_meta["first_events"][m]
            best = best_within(first, budget)
            assert best is not None, f"{arm_name} / {seed}: ilk tur bütçe dışında"
            runs.append(dict(seed=seed, events=first, first_events=first, continuation_events=[],
                             best_at_budget=int(best), wall_s=0.0, returncode=None, timed_out=False))
        return dict(runs=runs, best_at_budget=min(r["best_at_budget"] for r in runs),
                    continuation_wall_s=0.0, conceptual_prelude_s=float(prefix),
                    continuation_budget_s=max(0.0, float(remaining)), skipped_due_budget=True)
    common = instance_common(inst)
    start = time.time()
    raw = parallel_tasks(T.task_guided, [dict(common, budget=remaining, seed=SEEDS[m],
                                             cand=candidate_lists[m], init_tour=phase["tours"][m],
                                             timeout=remaining + PAD_SECONDS)
                                          for m in range(W)])
    runs = []
    for m, (seed, res) in enumerate(zip(SEEDS, raw)):
        assert (res["returncode"] in (0, None) and
                (res["returncode"] == 0 or res["timed_out"])), \
            f"{arm_name} / {seed}: LKH çıkış kodu {res['returncode']}"
        first = phase_meta["first_events"][m]
        resumed = event_list(res, prefix + res["t_start"] - start)
        if res["cost_check"] is not None and res["returncode"] == 0:
            resumed.append([int(res["cost_check"]),
                            float(prefix + res["t_start"] - start + res["wall"])])
        events = sorted(first + resumed, key=lambda e: e[1])
        best = best_within(events, budget)
        assert best is not None, f"{arm_name} / {seed}: bütçe içinde tur yok"
        runs.append(dict(seed=seed, events=events, first_events=first, continuation_events=resumed,
                         best_at_budget=int(best), wall_s=float(res["wall"]),
                         returncode=res["returncode"], timed_out=bool(res["timed_out"])))
    return dict(runs=runs, best_at_budget=min(r["best_at_budget"] for r in runs),
                continuation_wall_s=float(time.time() - start),
                conceptual_prelude_s=float(prefix), continuation_budget_s=float(remaining))

def load_phase(stage_dir):
    meta_path, array_path = stage_dir / "phase1.json", stage_dir / "phase1.npz"
    if meta_path.exists() and array_path.exists():
        with np.load(array_path, allow_pickle=False) as z:
            arrays = {k: z[k] for k in z.files}
        return arrays, json.loads(meta_path.read_text())
    return None

def load_lists(stage_dir, name):
    meta_path, array_path = stage_dir / "candidates.json", OUTPUT_ROOT / "candidates" / (name + ".npz")
    if meta_path.exists() and array_path.exists():
        with np.load(array_path, allow_pickle=False) as z:
            arrays = {k: z[k] for k in z.files}
        return arrays, json.loads(meta_path.read_text())
    return None

def run_one_instance(inst, ordinal):
    name = inst["name"]
    final_path = OUTPUT_ROOT / "instances" / (name + ".json")
    if final_path.exists():
        data = json.loads(final_path.read_text())
        assert data.get("complete") is True and set(data["arms"]) == set(ARM_NAMES)
        print("ATLA (tamam):", name, flush=True)
        return data
    stage_dir = OUTPUT_ROOT / "stages" / name
    session_marker = stage_dir / "runtime_session_id.txt"
    if stage_dir.exists() and any(stage_dir.iterdir()):
        old_id = session_marker.read_text().strip() if session_marker.exists() else None
        if old_id != SESSION_ID:
            print("Yeni Colab VM: yarım kalan örnek aynı donanımda baştan çalıştırılıyor:", name, flush=True)
            shutil.rmtree(stage_dir)
            (OUTPUT_ROOT / "candidates" / (name + ".npz")).unlink(missing_ok=True)
    stage_dir.mkdir(parents=True, exist_ok=True)
    session_marker.write_text(SESSION_ID)
    xy_sha = hashlib.sha256(np.ascontiguousarray(inst["xy"]).tobytes()).hexdigest()
    controls = ["restart_pop_5", "augment_nearest5p1",
                "augment_frequency5p1", "augment_betatour5p1"]
    rng = np.random.default_rng(20000 + ordinal)
    control_order = [str(x) for x in rng.permutation(controls)]
    component_order = [str(x) for x in rng.permutation(
        ["pop5_full", "pop6_full", "controls"])]

    def ensure_full(arm, k):
        path = stage_dir / (arm + ".json")
        if not path.exists():
            print(name, "/ kesintisiz POPMUSIC", k, flush=True)
            write_json_atomic(path, make_full_arm(inst, k))

    def ensure_controls():
        phase_loaded = load_phase(stage_dir)
        if phase_loaded is None:
            print(name, "/ altı ortak ilk deneme", flush=True)
            phase, phase_meta = make_phase1(inst)
            write_npz_atomic(stage_dir / "phase1.npz", **phase)
            write_json_atomic(stage_dir / "phase1.json", phase_meta)
        else:
            phase, phase_meta = phase_loaded
        lists_loaded = load_lists(stage_dir, name)
        if lists_loaded is None:
            print(name, "/ aday listeleri", flush=True)
            lists, list_meta = make_candidate_lists(inst, phase)
            write_npz_atomic(OUTPUT_ROOT / "candidates" / (name + ".npz"), **lists)
            write_json_atomic(stage_dir / "candidates.json", list_meta)
        else:
            lists, list_meta = lists_loaded
        for arm in control_order:
            path = stage_dir / (arm + ".json")
            if path.exists():
                continue
            print(name, "/", arm, flush=True)
            result = make_resumed_arm(inst, phase, phase_meta, lists[arm],
                                      list_meta[arm + "_overhead_s"], arm)
            write_json_atomic(path, result)
        return phase_meta, list_meta

    for component in component_order:
        if component == "controls":
            phase_meta, list_meta = ensure_controls()
        elif component == "pop5_full":
            ensure_full(component, K_POP5)
        elif component == "pop6_full":
            ensure_full(component, K_AUG6)
        else:
            raise AssertionError(component)
    arms = {arm: json.loads((stage_dir / (arm + ".json")).read_text()) for arm in ARM_NAMES}
    first_match = [int(arms["pop5_full"]["runs"][m]["events"][0][0]) ==
                   int(phase_meta["first_events"][m][0][0]) for m in range(W)]
    data = dict(schema=3, complete=True, name=name, group=inst["group"], n=len(inst["xy"]),
                budget_s=float(inst["budget"]), optimum=inst["optimum"], ew=inst["ew"],
                xy_sha256=xy_sha, seeds=SEEDS, control_order=control_order,
                component_order=component_order, hardware=HARDWARE,
                phase1=phase_meta, overheads=list_meta,
                first_trial_matches_pop5_full=first_match, arms=arms)
    write_json_atomic(final_path, data)
    b = {arm: arms[arm]["best_at_budget"] for arm in ARM_NAMES}
    print(f"TAMAM {name}: POP5={b['pop5_full']}, POP6={b['pop6_full']}, "
          f"restart5={b['restart_pop_5']}, nearest5+1={b['augment_nearest5p1']}, "
          f"frequency5+1={b['augment_frequency5p1']}, "
          f"BetaTour5+1={b['augment_betatour5p1']}", flush=True)
    return data


## 5. Çalıştırma ve kaldığı yerden sürdürme

Her örnek ve aşama Drive'a atomik kaydedilir. Tamamlanan örnekler atlanır. Aynı Colab sanal makinesinde kesilen aşamadan sürer; yeni sanal makinede yarım kalan tek örnek süre tutarlılığı için yeniden çalışır. Hata ayrıntıları `errors/` klasörüne yazılır. `MAX_NEW_INSTANCES` ile bir oturumda işlenecek yeni örnek sayısını sınırlayabilirsiniz. Tam 40 örnek ve ilk-tur denetimleri tamamlanmadan çıkarımsal p-değerleri verilmez.


In [ ]:
def warmup():
    xy = T.to_int_coords(T.gen_instance(100, "uniform", np.random.default_rng(9090)))
    succ = np.stack([np.roll(np.arange(100), -1)] * W, axis=1)
    M.predict_tourfold(MODEL, xy, K_GRAPH, DEVICE, ens_succ=succ, ens_cost=np.ones(W))
    first = T.task_first_trial(LKH, str(WORK_ROOT / "lkh_tmp"), SEEDS[0], xy, k=K_POP5,
                               timeout=30)
    assert T.is_valid_tour(first["tour"], 100) and first["cand"].shape == (100, K_POP5)
    print("GPU ve LKH öz sınaması tamam.")

warmup()

def save_progress(last_instance=None, state="running"):
    finished = sorted(p.stem for p in (OUTPUT_ROOT / "instances").glob("*.json"))
    write_json_atomic(OUTPUT_ROOT / "progress.json",
                      dict(state=state, completed=len(finished), total=len(INSTANCES),
                           finished=finished, last_instance=last_instance,
                           updated_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())))

save_progress(state="running")
session_start = time.time()
newly_finished = 0
for ordinal, inst in enumerate(INSTANCES):
    final_file = OUTPUT_ROOT / "instances" / (inst["name"] + ".json")
    if final_file.exists():
        record = json.loads(final_file.read_text())
        assert record.get("complete") and record.get("xy_sha256") == hashlib.sha256(
            np.ascontiguousarray(inst["xy"]).tobytes()).hexdigest(), \
            f"{inst['name']}: eski kayıt mevcut kurulumla uyuşmuyor"
        print("ATLA (tamam):", inst["name"], flush=True)
        continue
    if MAX_NEW_INSTANCES is not None and newly_finished >= MAX_NEW_INSTANCES:
        print("Bu oturum için MAX_NEW_INSTANCES sınırına ulaşıldı.")
        break
    try:
        run_one_instance(inst, ordinal)
        newly_finished += 1
        save_progress(last_instance=inst["name"])
    except Exception as exc:
        failure = dict(instance=inst["name"], error=repr(exc), traceback=traceback.format_exc(),
                       utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
        write_json_atomic(OUTPUT_ROOT / "errors" / (inst["name"] + ".json"), failure)
        save_progress(last_instance=inst["name"], state="error")
        raise

done = len(list((OUTPUT_ROOT / "instances").glob("*.json")))
save_progress(state="complete" if done == len(INSTANCES) else "partial")
print(f"Bu oturumda yeni {newly_finished}, toplam {done}/{len(INSTANCES)} örnek tamamlandı. "
      f"Süre: {(time.time()-session_start)/3600:.2f} saat. Kayıt: {OUTPUT_ROOT}")


GPU ve LKH öz sınaması tamam.
augment_clu10k_000 / kesintisiz POPMUSIC 6
augment_clu10k_000 / kesintisiz POPMUSIC 5
augment_clu10k_000 / altı ortak ilk deneme
augment_clu10k_000 / aday listeleri
augment_clu10k_000 / augment_nearest5p1
augment_clu10k_000 / augment_frequency5p1
augment_clu10k_000 / restart_pop_5
augment_clu10k_000 / augment_betatour5p1
TAMAM augment_clu10k_000: POP5=33760016, POP6=33760199, restart5=33760703, nearest5+1=33761006, frequency5+1=33760877, BetaTour5+1=33760774
augment_clu10k_001 / kesintisiz POPMUSIC 6
augment_clu10k_001 / kesintisiz POPMUSIC 5
augment_clu10k_001 / altı ortak ilk deneme
augment_clu10k_001 / aday listeleri
augment_clu10k_001 / augment_frequency5p1
augment_clu10k_001 / augment_betatour5p1
augment_clu10k_001 / augment_nearest5p1
augment_clu10k_001 / restart_pop_5
TAMAM augment_clu10k_001: POP5=16202132, POP6=16201186, restart5=16202082, nearest5+1=16203506, frequency5+1=16204260, BetaTour5+1=16202181
augment_clu10k_002 / altı ortak ilk deneme
a

## 6. Analiz ve aday denetimi

`summary/per_instance.csv` örnek sonuçlarını, `per_seed.csv` tohum sonuçlarını, `candidate_diagnostics.csv` aday ve süre göstergelerini içerir. `paired_stats.csv` kazanım/beraberlik/kayıp sayılarını, ortalama yüzde farkları, kesin işaret p-değerlerini ve ortalama fark için bootstrap aralığını verir. `RAPOR.md` iki birincil kararı ve sınırlamaları açıkça yazar.

Bu deney altıncı kenarın aday dosyasında bulunduğunu doğrular; LKH araması içinde kaç kez denendiği bu kayıtlardan çıkarılamaz. Mevcut TourFold modeli ilk turlara ihtiyaç duyduğu için 5+1 kolları LKH'yi yeniden başlatır. Kesintisiz POPMUSIC kıyasları bu pratik maliyeti de kapsar.


In [ ]:
from scipy.stats import binomtest
from IPython.display import display


TREATMENT = "augment_betatour5p1"
CO_PRIMARY = ("pop5_full", "pop6_full")
SECONDARY = ("restart_pop_5", "augment_nearest5p1", "augment_frequency5p1")
SIX_SLOT_ARMS = ("augment_nearest5p1", "augment_frequency5p1", TREATMENT)


def first_match_flags(item):
    """The common POP5 first trials are checked against continuous POP5 only."""
    flags = item.get("first_trial_matches_pop5_full")
    if flags is None:
        flags = item.get("first_trial_matches_uninterrupted")
    assert isinstance(flags, list) and len(flags) == W
    assert all(isinstance(value, bool) for value in flags)
    return flags


def load_finished():
    expected = {item["name"]: item for item in INSTANCES}
    assert len(expected) == len(INSTANCES), "Duplicate planned instance name"
    records, seen = [], set()
    for path in sorted((OUTPUT_ROOT / "instances").glob("*.json")):
        item = json.loads(path.read_text())
        name = item.get("name")
        assert name in expected, f"Unexpected instance record: {path}"
        assert path.stem == name, f"Result filename/name mismatch: {path}"
        assert name not in seen, f"Duplicate instance record: {name}"
        seen.add(name)
        assert item.get("complete") is True, f"Incomplete final record: {name}"
        assert set(item["arms"]) == set(ARM_NAMES)
        assert item["seeds"] == SEEDS
        assert first_match_flags(item)
        for arm in ARM_NAMES:
            runs = item["arms"][arm]["runs"]
            assert len(runs) == W and [run["seed"] for run in runs] == SEEDS
            assert int(item["arms"][arm]["best_at_budget"]) == min(
                int(run["best_at_budget"]) for run in runs)
        expected_xy = hashlib.sha256(
            np.ascontiguousarray(expected[name]["xy"]).tobytes()).hexdigest()
        assert item["xy_sha256"] == expected_xy, f"Coordinate hash mismatch: {name}"
        records.append(item)
    return records


def candidate_keys(n, neighbours):
    neighbours = np.asarray(neighbours, dtype=np.int64)
    assert neighbours.ndim == 2 and neighbours.shape[0] == n
    source = np.arange(n, dtype=np.int64)[:, None]
    assert np.all((neighbours >= 0) & (neighbours < n))
    assert not np.any(neighbours == source)
    return np.minimum(source, neighbours) * np.int64(n) + np.maximum(source, neighbours)


def audit_candidate_array(candidates, n, slots):
    assert candidates.shape == (W, n, slots), candidates.shape
    assert np.issubdtype(candidates.dtype, np.integer)
    assert np.all((candidates >= 0) & (candidates < n))
    assert not np.any(candidates == np.arange(n)[None, :, None])
    ordered = np.sort(candidates, axis=2)
    assert not np.any(ordered[:, :, 1:] == ordered[:, :, :-1]), "Repeated candidate"


def verify_saved_candidates(item):
    """Audit the exact candidate arrays retained for every completed instance."""
    n, name = int(item["n"]), item["name"]
    path = OUTPUT_ROOT / "candidates" / (name + ".npz")
    assert path.exists(), f"Missing candidate array: {path}"
    with np.load(path, allow_pickle=False) as z:
        assert set(("restart_pop_5",) + SIX_SLOT_ARMS).issubset(z.files)
        pop5 = z["restart_pop_5"]
        audit_candidate_array(pop5, n, 5)
        for arm in SIX_SLOT_ARMS:
            augmented = z[arm]
            audit_candidate_array(augmented, n, 6)
            assert np.array_equal(augmented[:, :, :5], pop5), \
                f"POP5 prefix changed: {name}, {arm}"
        for seed_index, seed in enumerate(SEEDS):
            old = np.unique(candidate_keys(n, pop5[seed_index]))
            for arm in SIX_SLOT_ARMS:
                added = candidate_keys(n, z[arm][seed_index, :, 5, None]).ravel()
                assert not np.isin(added, old).any(), \
                    f"Non-novel sixth edge: {name}, seed {seed}, {arm}"
    return True


records = load_finished()
for item in records:
    verify_saved_candidates(item)
mismatch = [r["name"] for r in records if not all(first_match_flags(r))]

COMPARISONS = [
    ("BetaTour 5+1 / continuous POP5 [co-primary]", TREATMENT, "pop5_full", "co_primary"),
    ("BetaTour 5+1 / continuous POP6 [co-primary]", TREATMENT, "pop6_full", "co_primary"),
    ("BetaTour 5+1 / restarted POP5", TREATMENT, "restart_pop_5", "secondary"),
    ("BetaTour 5+1 / nearest 5+1", TREATMENT, "augment_nearest5p1", "secondary"),
    ("BetaTour 5+1 / frequency 5+1", TREATMENT, "augment_frequency5p1", "secondary"),
]

rows, seed_rows, diagnostic_rows = [], [], []
for item in records:
    budget = float(item["budget_s"])
    row = dict(instance=item["name"], group=item["group"], n=item["n"],
               budget_s=budget, first_match_all=all(first_match_flags(item)),
               phase1_s=item["phase1"]["wall_s"],
               cpu_model=item["hardware"]["cpu_model"], gpu=item["hardware"]["gpu"])
    for arm in ARM_NAMES:
        arm_data = item["arms"][arm]
        row[arm] = int(arm_data["best_at_budget"])
        for run in arm_data["runs"]:
            seed_rows.append(dict(instance=item["name"], group=item["group"],
                                  n=item["n"], seed=run["seed"], arm=arm,
                                  cost=int(run["best_at_budget"]), budget_s=budget))
    for label, treatment, control, kind in COMPARISONS:
        column = treatment + "_vs_" + control
        row[column + "_abs"] = row[treatment] - row[control]
        row[column + "_pct"] = 100.0 * (row[treatment] - row[control]) / row[control]
    rows.append(row)
    diag = dict(instance=item["name"], group=item["group"], n=item["n"],
                phase1_s=item["phase1"]["wall_s"])
    diag.update(item["overheads"])
    diagnostic_rows.append(diag)

summary_dir = OUTPUT_ROOT / "summary"
summary_dir.mkdir(parents=True, exist_ok=True)
per_instance = pd.DataFrame(rows)
per_seed = pd.DataFrame(seed_rows)
diagnostics = pd.DataFrame(diagnostic_rows)
per_instance.to_csv(summary_dir / "per_instance.csv", index=False)
per_seed.to_csv(summary_dir / "per_seed.csv", index=False)
diagnostics.to_csv(summary_dir / "candidate_diagnostics.csv", index=False)


def paired_stats(values, bootstrap_seed, inferential):
    values = np.asarray(values, dtype=float)
    assert len(values) and np.isfinite(values).all()
    wins = int((values < 0).sum())
    ties = int((values == 0).sum())
    losses = int((values > 0).sum())
    if inferential:
        p = float(binomtest(wins, wins + losses, 0.5).pvalue) if wins + losses else 1.0
        rng = np.random.default_rng(bootstrap_seed)
        draw = rng.integers(0, len(values), size=(5000, len(values)))
        means = values[draw].mean(axis=1)
        lo, hi = (float(q) for q in np.quantile(means, [0.025, 0.975]))
    else:
        p, lo, hi = np.nan, np.nan, np.nan
    return dict(instances=len(values), win=wins, tie=ties, loss=losses,
                mean_delta_pct=float(values.mean()),
                median_delta_pct=float(np.median(values)),
                ci95_low_pct=lo, ci95_high_pct=hi, sign_p=p)


inference_ready = len(records) == len(INSTANCES) == 40 and not mismatch
stats_rows = []
if len(per_instance):
    groups = [("all", per_instance)] + list(per_instance.groupby("group", sort=True))
    for group, frame in groups:
        for i, (label, treatment, control, kind) in enumerate(COMPARISONS):
            column = treatment + "_vs_" + control + "_pct"
            stats_rows.append(dict(group=group, comparison=label, class_of_comparison=kind,
                                   **paired_stats(frame[column].to_numpy(), 20261027 + i,
                                                  inferential=(group == "all" and inference_ready))))
stats = pd.DataFrame(stats_rows)
co_primary_success = None
if len(stats):
    stats["sign_p_holm_secondary"] = np.nan
    secondary_ix = stats.index[(stats["group"] == "all") &
                               (stats["class_of_comparison"] == "secondary")].tolist()
    if inference_ready:
        assert len(secondary_ix) == len(SECONDARY)
        ordered = sorted(secondary_ix, key=lambda j: stats.loc[j, "sign_p"])
        running = 0.0
        for rank, j in enumerate(ordered):
            running = max(running, (len(ordered) - rank) * float(stats.loc[j, "sign_p"]))
            stats.loc[j, "sign_p_holm_secondary"] = min(1.0, running)
        primary = stats[(stats["group"] == "all") &
                        (stats["class_of_comparison"] == "co_primary")]
        assert len(primary) == len(CO_PRIMARY)
        co_primary_success = bool(all(
            (r.sign_p < 0.05) and (r.win > r.loss) and (r.mean_delta_pct < 0)
            for r in primary.itertuples()))
stats.to_csv(summary_dir / "paired_stats.csv", index=False)

if mismatch:
    note = (f"INVALID COMPARISON: POP5 first-trial costs disagree on {len(mismatch)} "
            "instances. Inferential results are withheld; raw data remain available.")
elif not inference_ready:
    note = f"PARTIAL RESULT: {len(records)}/40 instances complete; no inferential results yet."
else:
    note = ("Complete: 40/40 new 10k/20k synthetic instances. "
            f"Co-primary practical success: {co_primary_success}.")

cpu_models = sorted({r["hardware"]["cpu_model"] for r in records})
lines = ["# BetaTour POPMUSIC 5+1 augmentation experiment", "", note, "",
         "Each six-slot augmented list preserves all five seed-specific POPMUSIC-5 entries in order.",
         "The added sixth edge is absent from the entire undirected POPMUSIC-5 graph.",
         "BetaTour, frequency and nearest controls share the five POP entries and sixth-edge alpha=500.",
         "Percent difference = 100 * (BetaTour - control) / control; negative favours BetaTour.",
         "Co-primary practical comparisons: BetaTour 5+1 versus continuous POPMUSIC-5 and POPMUSIC-6.",
         "Success requires BOTH exact two-sided sign-test p-values < 0.05 and BetaTour-favouring direction (more wins than losses and negative mean difference).",
         "This intersection-union decision requires no multiplicity correction across the two co-primary tests.",
         "The three secondary overall sign tests (restarted POP5, nearest 5+1, frequency 5+1) receive Holm correction.",
         "Subgroup rows are descriptive only. Bootstrap intervals describe mean paired differences, not sign-test p-values.",
         "The six first tours are shared across the four restarted arms and are checked against continuous POP5; continuous POP6 can have different first tours.",
         "Common first phase, arm-specific preparation and continuation count against each deadline. Drive writes and archive export do not.",
         "This holdout covers synthetic uniform and clustered 10k/20k instances; it does not establish effects on TSPLIB or other distributions.",
         "Incomplete data or POP5 first-trial mismatches suppress all inferential results.", ""]
if len(cpu_models) > 1:
    lines += ["WARNING: instances used multiple CPU models: " + ", ".join(cpu_models), ""]
if mismatch:
    lines += ["POP5 first-trial mismatches: " + ", ".join(mismatch), ""]
if len(stats):
    lines += ["```", stats.round(6).to_string(index=False), "```", ""]
lines += ["Raw files: summary/per_instance.csv, summary/per_seed.csv, summary/candidate_diagnostics.csv.",
          "The separate audit ZIP in Drive includes stages/phase1.npz and candidates/*.npz for candidate-list auditing."]
(summary_dir / "RAPOR.md").write_text("\n".join(lines), encoding="utf-8")
print(note)
if mismatch:
    print("POP5 first-trial mismatches:", mismatch)
if len(stats):
    display(stats[stats["group"] == "all"].round(6))


Complete: 40/40 new 10k/20k synthetic instances. Co-primary practical success: False.


,group,comparison,class_of_comparison,instances,win,tie,loss,mean_delta_pct,median_delta_pct,ci95_low_pct,ci95_high_pct,sign_p,sign_p_holm_secondary
0,all,BetaTour 5+1 / continuous POP5 [co-primary],co_primary,40,12,0,28,0.008588,0.004070,0.002224,0.014514,0.016589,NaN
1,all,BetaTour 5+1 / continuous POP6 [co-primary],co_primary,40,9,0,31,0.012508,0.005613,0.007283,0.018965,0.000680,NaN
2,all,BetaTour 5+1 / restarted POP5,secondary,40,22,0,18,-0.001242,-0.002321,-0.004878,0.002710,0.635828,0.635828
3,all,BetaTour 5+1 / nearest 5+1,secondary,40,32,0,8,-0.007316,-0.008020,-0.013742,-0.000342,0.000182,0.000546
4,all,BetaTour 5+1 / frequency 5+1,secondary,40,28,0,12,-0.007707,-0.002587,-0.015649,-0.001921,0.016589,0.033178


## 7. Doğrulanmış ZIP'ler ve otomatik indirme

Küçük `augment40_v1_results.zip` manifest, 40 örnek JSON'u, özet CSV'ler, rapor, aşama metaverisi ve denetim arşivinin SHA256 bilgisini içerir. 40/40 tamamlanınca tarayıcıya otomatik iner. Büyük `augment40_v1_audit.zip` ayrıca `stages/phase1.npz` ile tüm `candidates/*.npz` dizilerini içerir ve Drive'da saklanır. ZIP'ler önce yerelde kapatılır, tüm girişlerin CRC'si test edilir, sonra Drive'a atomik kopyalanır. Tarayıcı indirmesi yarım kalırsa Drive'daki dosyayı yeniden indirebilirsiniz.


In [ ]:
import shutil
import zipfile


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        while True:
            chunk = source.read(4 * 1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def build_checked_zip(filename, include_arrays):
    """Write locally, test every entry, then atomically publish to Drive."""
    local = WORK_ROOT / filename
    local_tmp = WORK_ROOT / (filename + ".tmp")
    count = 0
    with zipfile.ZipFile(local_tmp, "w", compression=zipfile.ZIP_DEFLATED,
                         compresslevel=6, allowZip64=True) as archive:
        for path in sorted(OUTPUT_ROOT.rglob("*")):
            if not path.is_file() or path.suffix in (".tmp", ".zip") or path.name.endswith(".tmp.npz"):
                continue
            if include_arrays and path.name == "archive_info.json":
                continue
            if not include_arrays and path.suffix == ".npz":
                continue
            archive.write(path, str(path.relative_to(OUTPUT_ROOT)),
                          compress_type=zipfile.ZIP_STORED if path.suffix == ".npz"
                          else zipfile.ZIP_DEFLATED)
            count += 1
    with zipfile.ZipFile(local_tmp) as check:
        assert check.testzip() is None, f"ZIP CRC check failed: {filename}"
        assert len(check.namelist()) == count
    os.replace(local_tmp, local)
    target = OUTPUT_ROOT / filename
    drive_tmp = OUTPUT_ROOT / (filename + ".tmp")
    shutil.copy2(local, drive_tmp)
    os.replace(drive_tmp, target)
    assert target.stat().st_size == local.stat().st_size
    assert zipfile.is_zipfile(target), f"Published ZIP is incomplete: {target}"
    local_sha = file_sha256(local)
    assert file_sha256(target) == local_sha, f"Published ZIP differs from local ZIP: {target}"
    return dict(path=str(target), bytes=local.stat().st_size,
                sha256=local_sha, files=count, local=str(local))


progress = json.loads((OUTPUT_ROOT / "progress.json").read_text())
complete = progress["state"] == "complete" and progress["completed"] == progress["total"] == 40
audit_info = None
if complete and INCLUDE_ARRAYS_IN_ZIP:
    audit_info = build_checked_zip(EXPERIMENT_NAME + "_audit.zip", include_arrays=True)
    print(f"Tam denetim ZIP'i hazır: {audit_info['path']} "
          f"({audit_info['bytes'] / 1024**2:.1f} MB; SHA256 {audit_info['sha256']})")

write_json_atomic(OUTPUT_ROOT / "archive_info.json",
                  dict(experiment=EXPERIMENT_NAME, complete=complete,
                       audit_archive=audit_info,
                       note="Results ZIP downloads automatically; full candidate/phase arrays stay in Drive audit ZIP."))
results_info = build_checked_zip(EXPERIMENT_NAME + "_results.zip", include_arrays=False)
print(f"Sonuç ZIP'i: {results_info['path']} "
      f"({results_info['bytes'] / 1024**2:.1f} MB; SHA256 {results_info['sha256']})")
if SAVE_TO_DRIVE and complete:
    from google.colab import files
    files.download(results_info["local"])
elif SAVE_TO_DRIVE:
    print(f"Ara ZIP Drive'a kaydedildi ({progress['completed']}/{progress['total']}). "
          "40/40 tamamlanınca sonuç ZIP'i otomatik inecek.")


Tam denetim ZIP'i hazır: /content/drive/MyDrive/BetaTour/augment40_v1/augment40_v1_audit.zip (288.9 MB; SHA256 fd0a3be4a38efdb5ec8086e468abf8bb437c3ae5e507f4c7b17fd696c41d5236)
Sonuç ZIP'i: /content/drive/MyDrive/BetaTour/augment40_v1/augment40_v1_results.zip (1.7 MB; SHA256 00174a890bd74d3209d5efbc3f8115e9dcb2e8dfe0d3e19fda7b0aa6858b3caa)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>